# 🚀 Enterprise-Grade XLM-RoBERTa Urgency Classifier
This notebook contains advanced techniques for maximum accuracy:
- **Class Imbalance Handling:** Uses dynamic Class Weights so rare CRITICAL events are heavily penalized if missed.
- **Early Stopping:** Monitors validation loss and stops training automatically to prevent overfitting.
- **Hyperparameter Tuning:** Optimized learning rate, weight decay, and warmup steps.
- **Longer Training:** Set to 15 epochs max (will stop early if it peaks).

In [ ]:
!pip install transformers datasets accelerate loguru scikit-learn pandas torch

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
import joblib
import os

# 1. Load Data
df = pd.read_csv('complaints_processed.csv')
df = df.dropna(subset=['clean_text', 'urgency'])

# Add some synthetic variation by duplicating the dataset with slight noise (optional augmentation step)
# For now, we will maximize the existing data's potential.

# 2. Prepare Labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['urgency'])

train_df, eval_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# 3. Compute Class Weights (Crucial for Class Imbalance!)
class_weights = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(train_df['label']), 
    y=train_df['label']
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Computed Class Weights: {class_weights_tensor}")

# 4. Tokenize
model_name = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'}))
eval_ds = Dataset.from_pandas(eval_df[['clean_text', 'label']].rename(columns={'clean_text': 'text'}))

train_ds = train_ds.map(tokenize_fn, batched=True).remove_columns(['text', '__index_level_0__']).with_format('torch')
eval_ds = eval_ds.map(tokenize_fn, batched=True).remove_columns(['text', '__index_level_0__']).with_format('torch')

# 5. Define Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(le.classes_))

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1_score(labels, preds, average='weighted')}

# 6. Custom Trainer to inject Class Weights
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# 7. Enterprise Training Arguments
args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=15,             # Train much longer
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,     # Keep the best weights, not the last
    metric_for_best_model='f1',
    learning_rate=2e-5,              # Smaller, careful learning rate
    weight_decay=0.01,               # Prevent overfitting
    warmup_ratio=0.1,
    fp16=True
)

trainer = CustomTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # Stop if no improvement after 3 epochs
)

print("🚀 Starting enterprise-grade training...")
trainer.train()

# 8. Save Model
os.makedirs('urgency_model', exist_ok=True)
model.save_pretrained('./urgency_model')
tokenizer.save_pretrained('./urgency_model')
joblib.dump(le, './urgency_model/label_encoder.pkl')

print('✅ Enterprise Training complete! Zip and download the `urgency_model` folder.')